## 0. Configuration & Setup
Defines the Unity Catalog namespace for the Gold presentation layer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "dbw_proman_lakehouse_01"
SCHEMA = "default"
BASE_PATH = f"{CATALOG}.{SCHEMA}"

### 1. Conformed Dimensions
Transforms Silver 3NF tables into Kimball dimensions optimized for a distributed Lakehouse, implementing Slowly Changing Dimension (SCD) Type 1 patterns to maintain the current operational state.

* **State Management (SCD Type 1):** Utilizes a modular Python function (`merge_dimension`) to execute Delta Lake `MERGE INTO` (upsert) operations for `dim_user`, `dim_project`, and `dim_task`. Unlike a full table overwrite, this approach only modifies specific rows that have changed, minimizing computational overhead and preserving a granular Delta transaction log.
* **Surrogate Keys:** Generates deterministic string keys (`user_key`, `project_key`, `task_key`) using MD5 hashing. Unlike sequential integers (which force a single-node processing bottleneck), cryptographic hashing allows Spark to evaluate and generate keys in parallel across distributed worker nodes, decoupling the warehouse from source IDs without sacrificing performance.
* **Business Logic:** Injects financial rate cards (`hourly_rate` based on user roles) and handles dynamic budget fallbacks.
* **Date Dimension:** Generates a standard static calendar with `yyyyMMdd` integer keys for cross-fact filtering, executed via a standard batch overwrite.

In [0]:
# 1. Merge function for SCD Type 1 Dimensions
def merge_dimension(df_source, table_name, join_key):
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        (
            delta_table.alias("target")
            .merge(
                df_source.alias("source"),
                f"target.{join_key} = source.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"Merged updates into {table_name}")
    else:
        # Day 0 baseline creation
        (
            df_source.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
        print(f"Created {table_name} baseline")

# 2. Transform and Merge dim_user
df_dim_user = (
    spark.table(f"{BASE_PATH}.silver_users")
    .select(
        F.md5(F.col("user_id").cast("string")).alias("user_key"),
        F.col("user_id"),
        F.col("full_name"),
        F.col("email"),
        F.col("department"),
        F.col("role"),
        F.when(F.col("role").like("%Senior%") | F.col("role").like("%Lead%"), 95.0)
         .when(F.col("role").like("%Engineer%") | F.col("role").like("%Developer%"), 80.0)
         .when(F.col("role").like("%Designer%"), 75.0)
         .when(F.col("role").like("%QA%"), 65.0)
         .otherwise(70.0).alias("hourly_rate"),
        F.col("is_active"),
        F.current_timestamp().alias("_loaded_at")
    )
)
merge_dimension(df_dim_user, f"{BASE_PATH}.dim_user", "user_key")

# 3. Transform and Merge dim_project
df_dim_project = (
    spark.table(f"{BASE_PATH}.silver_projects")
    .select(
        F.md5(F.col("project_id").cast("string")).alias("project_key"),
        F.col("project_id"),
        F.col("project_name"),
        F.col("status"),
        F.col("start_date"),
        F.col("due_date"),
        F.col("budget_hours"),
        F.coalesce(F.col("budget_amount"), F.col("budget_hours") * 75.0).alias("budget_amount"),
        F.current_timestamp().alias("_loaded_at")
    )
)
merge_dimension(df_dim_project, f"{BASE_PATH}.dim_project", "project_key")

# 4. Transform and Merge dim_task
df_dim_task = (
    spark.table(f"{BASE_PATH}.silver_tasks")
    .select(
        F.md5(F.col("task_id").cast("string")).alias("task_key"),
        F.col("task_id"),
        F.col("project_id"),
        F.col("assigned_user_id"),
        F.col("task_name"),
        F.col("priority"),
        F.col("status"),
        F.col("estimated_hours"),
        F.current_timestamp().alias("_loaded_at")
    )
)
merge_dimension(df_dim_task, f"{BASE_PATH}.dim_task", "task_key")

# 5. Overwrite Static dim_date (Kept as-is)
df_dim_date = (
    spark.sql("""
        SELECT explode(sequence(to_date('2026-01-01'), to_date('2026-12-31'), interval 1 day)) as calendar_date
    """)
    .select(
        F.date_format("calendar_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("calendar_date"),
        F.year("calendar_date").alias("year"),
        F.quarter("calendar_date").alias("quarter"),
        F.month("calendar_date").alias("month"),
        F.date_format("calendar_date", "MMMM").alias("month_name"),
        F.dayofmonth("calendar_date").alias("day_of_month"),
        F.date_format("calendar_date", "EEEE").alias("day_name"),
        F.when(F.dayofweek("calendar_date").isin([1, 7]), True).otherwise(False).alias("is_weekend")
    )
)
df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BASE_PATH}.dim_date")
print("Saved dim_date")

Saved dim_user
Saved dim_project
Saved dim_task
Saved dim_date


### 2. Fact Tables (Surrogate Key Lookups)
Builds immutable transactional logs (Timesheets, Expenses, Documents).
* **Key Joins:** Joins Silver operational data against Gold dimensions to retrieve and enforce surrogate keys.
* **Orphan Handling:** Uses `coalesce` to map missing dimension references to an "Unknown" (`"-1"`) surrogate key.
* **Degenerate Dimensions:** Retains natural source IDs (e.g., `entry_id`, `expense_id`) directly in the facts for operational auditability.

In [0]:
dim_u = spark.table(f"{BASE_PATH}.dim_user").select("user_key", "user_id", "hourly_rate")
dim_p = spark.table(f"{BASE_PATH}.dim_project").select("project_key", "project_id")
dim_t = spark.table(f"{BASE_PATH}.dim_task").select("task_key", "task_id")

# Fact Timesheet: Grain = 1 row per logged entry 
df_silver_tt = spark.table(f"{BASE_PATH}.silver_timetracking")

df_fact_timesheet = (
    df_silver_tt
    .join(dim_u, on="user_id", how="left")
    .join(dim_p, on="project_id", how="left")
    .join(dim_t, on="task_id", how="left")
    .select(
        F.col("entry_id"),                                                  # Degenerate business key
        F.coalesce(F.col("user_key"), F.lit("-1")).alias("user_key"),         # Foreign key
        F.coalesce(F.col("project_key"), F.lit("-1")).alias("project_key"),   # Foreign key
        F.coalesce(F.col("task_key"), F.lit("-1")).alias("task_key"),         # Foreign key
        F.date_format("work_date", "yyyyMMdd").cast("int").alias("date_key"),# Foreign key
        F.col("hours_worked"),                                               # Effort measure
        F.round(F.col("hours_worked") * F.coalesce(F.col("hourly_rate"), F.lit(70.0)), 2).alias("labor_cost"), # Cost measure
        F.col("progress_pct"),                                               # Progress milestone metric
        F.col("activity_description"),
        F.current_timestamp().alias("_loaded_at")
    )
)
df_fact_timesheet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BASE_PATH}.fact_timesheet")
print("Saved fact_timesheet")


# Fact Expenses: Grain = 1 row per vendor invoice sidecar (Non-Labor OpEx)
df_silver_exp = spark.table(f"{BASE_PATH}.silver_expenses")

df_fact_expenses = (
    df_silver_exp
    .join(dim_p, on="project_id", how="left")
    .select(
        F.col("expense_id"),                                                 # Degenerate business key
        F.coalesce(F.col("project_key"), F.lit("-1")).alias("project_key"),   # Foreign key
        F.date_format("expense_date", "yyyyMMdd").cast("int").alias("date_key"), # Foreign key
        F.col("category"),
        F.col("vendor_name"),
        F.col("amount").alias("expense_amount"),                             # Non-labor financial measure
        F.col("adls_raw_uri"),                                               # Audit link to PDF receipt
        F.current_timestamp().alias("_loaded_at")
    )
)
df_fact_expenses.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BASE_PATH}.fact_expenses")
print("Saved fact_expenses")


# Fact Project Documents: Grain = 1 row per deliverable file
df_silver_docs = spark.table(f"{BASE_PATH}.silver_documents")

df_fact_documents = (
    df_silver_docs
    .join(dim_p, on="project_id", how="left")
    .join(dim_t, on="task_id", how="left")
    .select(
        F.col("document_id"),                                                # Degenerate business key
        F.coalesce(F.col("project_key"), F.lit("-1")).alias("project_key"),   # Foreign key
        F.coalesce(F.col("task_key"), F.lit("-1")).alias("task_key"),         # Foreign key
        F.col("file_name"),
        F.col("file_type"),
        F.col("file_size_mb"),                                               # Storage measure
        F.lit(1).alias("document_count"),                                    # Count measure
        F.current_timestamp().alias("_loaded_at")
    )
)
df_fact_documents.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BASE_PATH}.fact_project_documents")
print("Saved fact_project_documents")

Saved fact_timesheet
Saved fact_expenses
Saved fact_project_documents


### 3. Task Execution Mart (Velocity Analytics)
Produces a granular view of active task delivery.
* **Health Categorization:** Uses SQL `WHEN` conditions to dynamically flag tasks as "Stalled Progress", "Effort Overrun", or "Blocked".
* **Efficiency Metrics:** Calculates metrics like `hours_per_progress_pct` to track individual developer velocity.

In [0]:
task_timesheet_agg = (
    spark.table(f"{BASE_PATH}.fact_timesheet")
    .filter(F.col("task_key") != "-1")
    .groupBy("task_key")
    .agg(
        F.round(F.sum("hours_worked"), 1).alias("actual_hours_logged"),
        F.round(F.sum("labor_cost"), 2).alias("total_labor_cost"),
        F.max("progress_pct").alias("latest_progress_pct"),
        F.count("entry_id").alias("total_work_sessions")
    )
)

df_task_overview = (
    spark.table(f"{BASE_PATH}.dim_task").alias("t")
    .join(spark.table(f"{BASE_PATH}.dim_project").alias("p"), on="project_id", how="left")
    .join(spark.table(f"{BASE_PATH}.dim_user").alias("u"), 
          F.col("t.assigned_user_id") == F.col("u.user_id"), how="left")
    .join(task_timesheet_agg, on="task_key", how="left")
    .select(
        F.col("t.task_key"),
        F.col("t.task_id"),
        F.col("t.task_name"),
        F.col("p.project_id"),
        F.col("p.project_name"),
        F.col("u.full_name").alias("assignee_name"),
        F.col("u.department").alias("assignee_department"),
        F.col("t.priority"),
        F.col("t.status").alias("task_status"),
        F.col("t.estimated_hours"),
        F.coalesce(F.col("actual_hours_logged"), F.lit(0.0)).alias("actual_hours_logged"),
        F.coalesce(F.col("latest_progress_pct"), F.lit(0)).alias("latest_progress_pct"),
        # Variance: Positive means hours overrun
        F.round(
            F.coalesce(F.col("actual_hours_logged"), F.lit(0.0)) - F.col("t.estimated_hours"), 1
        ).alias("hour_variance"),
        # Hours spent per 1% of reported progress
        F.round(
            F.coalesce(F.col("actual_hours_logged"), F.lit(0.0)) / 
            F.when(F.coalesce(F.col("latest_progress_pct"), F.lit(0)) == 0, 1)
             .otherwise(F.col("latest_progress_pct")), 2
        ).alias("hours_per_progress_pct"),
        # Health Categorization
        F.when(F.col("t.status") == "Done", "Completed")
         .when(F.col("t.status") == "Blocked", "Blocked / Impeded")
         .when(F.coalesce(F.col("actual_hours_logged"), F.lit(0.0)) > F.col("t.estimated_hours"), "Effort Overrun")
         .when((F.coalesce(F.col("latest_progress_pct"), F.lit(0)) < 30) & 
               (F.coalesce(F.col("actual_hours_logged"), F.lit(0.0)) > (F.col("t.estimated_hours") * 0.6)), "Stalled Progress")
         .otherwise("On Track").alias("execution_health"),
        F.current_timestamp().alias("_updated_at")
    )
)

df_task_overview.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BASE_PATH}.agg_task_overview")
print("Saved agg_task_overview")

Saved agg_task_overview


### 4. Analytical Mart (Project Burndown)
Creates a wide, current-state aggregated view for executive dashboards.
* **Pre-Aggregation:** Rolls up labor hours, non-labor expenses, and document sizes by project.
* **Variance Logic:** Computes real-time project health metrics (burn rates, financial variances, task completion percentages) for direct AI/BI Dashboard consumption.

In [0]:
hours_agg = (
    spark.table(f"{BASE_PATH}.fact_timesheet")
    .filter(F.col("project_key") != "-1")
    .groupBy("project_key")
    .agg(
        F.round(F.sum("hours_worked"), 2).alias("total_actual_hours"),
        F.round(F.sum("labor_cost"), 2).alias("total_labor_cost")
    )
)

expenses_agg = (
    spark.table(f"{BASE_PATH}.fact_expenses")
    .filter(F.col("project_key") != "-1")
    .groupBy("project_key")
    .agg(
        F.round(F.sum("expense_amount"), 2).alias("total_non_labor_cost")
    )
)

docs_agg = (
    spark.table(f"{BASE_PATH}.fact_project_documents")
    .filter(F.col("project_key") != "-1")
    .groupBy("project_key")
    .agg(
        F.round(F.sum("file_size_mb"), 2).alias("total_storage_mb"),
        F.sum("document_count").alias("total_documents")
    )
)


tasks_agg = (
    spark.table(f"{BASE_PATH}.agg_task_overview")
    .groupBy("project_id")
    .agg(
        F.count("task_id").alias("total_tasks"),
        F.sum(F.when(F.col("task_status") == "Done", 1).otherwise(0)).alias("completed_tasks"),
        F.sum(F.when(F.col("task_status") == "Blocked", 1).otherwise(0)).alias("blocked_tasks"),
        F.round(F.avg("latest_progress_pct"), 1).alias("avg_task_progress_pct") 
    )
)

df_gold_overview = (
    spark.table(f"{BASE_PATH}.dim_project")
    .join(hours_agg, on="project_key", how="left")
    .join(expenses_agg, on="project_key", how="left")
    .join(docs_agg, on="project_key", how="left")
    .join(tasks_agg, on="project_id", how="left")
    .select(
        F.col("project_key"),
        F.col("project_id"),
        F.col("project_name"),
        F.col("status"),
        # Effort controlling
        F.col("budget_hours"),
        F.coalesce(F.col("total_actual_hours"), F.lit(0.0)).alias("actual_hours"),
        F.round(F.col("budget_hours") - F.coalesce(F.col("total_actual_hours"), F.lit(0.0)), 2).alias("hours_variance"),
        F.round((F.coalesce(F.col("total_actual_hours"), F.lit(0.0)) / F.col("budget_hours")) * 100, 1).alias("hours_burn_rate_pct"),
        # Financial controlling
        F.col("budget_amount"),
        F.coalesce(F.col("total_labor_cost"), F.lit(0.0)).alias("labor_cost"),
        F.coalesce(F.col("total_non_labor_cost"), F.lit(0.0)).alias("non_labor_cost"),
        F.round(
            F.coalesce(F.col("total_labor_cost"), F.lit(0.0)) + 
            F.coalesce(F.col("total_non_labor_cost"), F.lit(0.0)), 2
        ).alias("total_actual_cost"),
        F.round(
            F.col("budget_amount") - (
                F.coalesce(F.col("total_labor_cost"), F.lit(0.0)) + 
                F.coalesce(F.col("total_non_labor_cost"), F.lit(0.0))
            ), 2
        ).alias("financial_variance"),
        F.round(
            ((F.coalesce(F.col("total_labor_cost"), F.lit(0.0)) + 
              F.coalesce(F.col("total_non_labor_cost"), F.lit(0.0))) / 
             F.col("budget_amount")) * 100, 1
        ).alias("cost_burn_rate_pct"),
        # Operational task delivery
        F.coalesce(F.col("total_tasks"), F.lit(0)).alias("total_tasks"),
        F.coalesce(F.col("completed_tasks"), F.lit(0)).alias("completed_tasks"),
        F.round(
            (F.coalesce(F.col("completed_tasks"), F.lit(0)) / 
             F.when(F.col("total_tasks") == 0, 1).otherwise(F.col("total_tasks"))) * 100, 1
        ).alias("task_completion_pct"),
        # Asset storage
        F.coalesce(F.col("total_documents"), F.lit(0)).alias("total_documents"),
        F.coalesce(F.col("total_storage_mb"), F.lit(0.0)).alias("total_storage_mb"),
        F.coalesce(F.col("avg_task_progress_pct"), F.lit(0.0)).alias("avg_task_progress_pct"),
        F.coalesce(F.col("blocked_tasks"), F.lit(0)).alias("blocked_tasks"),
        F.current_timestamp().alias("_updated_at")
    )
)

df_gold_overview.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BASE_PATH}.agg_project_overview")
print("Saved agg_project_overview")

Saved agg_project_overview


### 5. Historical Snapshot Fact (Time-Series Tracking)
Preserves historical states to enable trend analysis without complex SCD Type 2 logic.

* **Idempotent Partition Overwrite:** Uses `.mode("overwrite")` combined with Delta's `replaceWhere` option (`snapshot_date = current_date()`). This guarantees idempotent execution. It safely replaces only the current day's data to prevent duplicates on intraday runs, while permanently locking and preserving all historical snapshots.
* **Partition Pruning:** Partitions the underlying Delta table by `snapshot_date` to physically support the daily overwrite boundary and ensure downstream BI queries execute efficiently without full table scans.

In [0]:

df_project_snapshot = (
    spark.table(f"{BASE_PATH}.agg_project_overview")
    .withColumn("snapshot_date", F.current_date())
)

(
    df_project_snapshot.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("snapshot_date")
    .option("replaceWhere", "snapshot_date = current_date()")
    .option("mergeSchema", "true")
    .saveAsTable(f"{BASE_PATH}.fact_project_snapshot")
)

print("Overwrote snapshot partition for current date in fact_project_snapshot")

Appended to fact_project_snapshot


In [0]:
df_task_snapshot = (
    spark.table(f"{BASE_PATH}.agg_task_overview")
    .withColumn("snapshot_date", F.current_date())
)

(
    df_task_snapshot.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("snapshot_date")
    .option("replaceWhere", "snapshot_date = current_date()")
    .option("mergeSchema", "true")
    .saveAsTable(f"{BASE_PATH}.fact_task_snapshot")
)
print("Overwrote snapshot partition for current date in fact_task_snapshot")

Appended to fact_task_snapshot
